# NimbusPay Ticket Triage — Fine-Tuning Notebook

**Assignment:** VIMA3YA ML Intern — LLM Fine-Tuning  
**Model:** `Qwen/Qwen2.5-1.5B-Instruct`  
**Stack:** Unsloth + TRL SFTTrainer  
**Hardware:** Google Colab T4 (free tier)

---
**Run order:**  
1. Setup & install  
2. Mount Drive + upload data  
3. Part B — Baselines (B1 zero-shot, B2 rules-in-prompt)  
4. Part C — Fine-tune pipeline  
5. Part D — Controlled experiments (E1–E5)  
6. Part E — Error analysis  
7. Generate test predictions  

## 0. Setup

In [ ]:
# Install Unsloth (wheels for Colab T4 / CUDA 12.1)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
    'unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git',
    'trl>=0.8.6', 'peft>=0.10', 'transformers>=4.40',
    'accelerate>=0.29', 'bitsandbytes>=0.43'], check=True)
print('Install done')

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/nimbuspay_triage'
import os; os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted:', DRIVE_DIR)

In [ ]:
# Upload your 3 data files: train_clean.jsonl, train.jsonl, dev.jsonl, test_inputs.jsonl
# Option A: upload manually
from google.colab import files
print('Upload: train.jsonl, train_clean.jsonl, dev.jsonl, test_inputs.jsonl')
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(fname, 'wb') as f: f.write(data)
    print(f'  Saved {fname} ({len(data)/1024:.0f} KB)')

# Option B (if already on Drive): copy from Drive
# import shutil
# for f in ['train.jsonl','train_clean.jsonl','dev.jsonl','test_inputs.jsonl']:
#     shutil.copy(f'{DRIVE_DIR}/{f}', f)

In [ ]:
import json, re, os, csv, time
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import torch
from datasets import Dataset
from transformers import AutoTokenizer

# ── constants ────────────────────────────────────────────────────────────────
MODEL_ID  = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ   = 512     # set after measuring below
SEED      = 42

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_raw   = load_jsonl('train.jsonl')
train_clean = load_jsonl('train_clean.jsonl')
dev_rows    = load_jsonl('dev.jsonl')
test_rows   = load_jsonl('test_inputs.jsonl')
print(f'train_raw={len(train_raw)}  train_clean={len(train_clean)}  dev={len(dev_rows)}  test={len(test_rows)}')

### Measure sequence lengths to choose max_seq_length

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def row_to_text(row):
    """Apply the model's chat template to get the full training string."""
    return tokenizer.apply_chat_template(
        row['messages'], tokenize=False, add_generation_prompt=False)

lens = [len(tokenizer.encode(row_to_text(r))) for r in train_clean]
lens.sort()
p50  = lens[len(lens)//2]
p95  = lens[int(len(lens)*0.95)]
p99  = lens[int(len(lens)*0.99)]
pmax = lens[-1]
print(f'p50={p50}  p95={p95}  p99={p99}  max={pmax}')
print(f'Recommended MAX_SEQ = {p99 + 32}  (p99 + 32 token headroom)')
# Update MAX_SEQ based on measurement:
MAX_SEQ = p99 + 32
print(f'Using MAX_SEQ = {MAX_SEQ}')

---
## Part B — Baselines

In [ ]:
from unsloth import FastLanguageModel

# Load base model in 4-bit (saves memory; fine for inference)
base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ,
    dtype=None,          # auto
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
print('Base model loaded')

In [ ]:
SYSTEM_SHORT = 'You are NimbusPay\'s ticket triage model. Read the ticket and return the triage JSON.'

RULES_SYSTEM = '''You are NimbusPay's ticket triage model. Read the ticket and return only a JSON object with exactly these 8 keys:
{"category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"}

category: payment_failed | refund | fraud | account_access | kyc | offers | other
  fraud wins over all others when a transaction the customer did not make is mentioned.
priority (apply in order):
  P1 if fraud, or amount>=50000 INR, or legal threat (RBI/ombudsman/court/lawyer)
  P2 if (payment_failed|refund) and amount>=5000 INR, or account_access
  P3 otherwise
amount_paise: integer paise (1 INR=100 paise), null if not stated. k=1000, lakh=100000.
txn_date: YYYY-MM-DD of the transaction (resolve relative dates). null if unknown.
txn_id: NP + 10 digits. null if none. For account_access/kyc/other: always null.
channel: upi|card|netbanking|wallet only if explicitly stated, else null.
language: hi-en if Hinglish (Hindi in Latin script), else en.
needs_human: true if P1 or customer says they contacted support before, else false.
Return ONLY the JSON object, no extra text.'''

def run_inference(model, tok, rows, system_msg, max_new_tokens=200, batch_size=8):
    """Run greedy inference; returns list of {id, output} dicts."""
    results = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        # Build prompts: use system_msg override + original user message
        prompts = []
        for r in batch:
            msgs = [
                {'role': 'system', 'content': system_msg},
                {'role': 'user',   'content': r['messages'][1]['content']},
            ]
            text = tok.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
            prompts.append(text)
        inputs = tok(prompts, return_tensors='pt', padding=True,
                     truncation=True, max_length=MAX_SEQ).to('cuda')
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # greedy
                pad_token_id=tok.eos_token_id,
            )
        for j, r in enumerate(batch):
            gen = out_ids[j][inputs['input_ids'].shape[1]:]
            text = tok.decode(gen, skip_special_tokens=True).strip()
            results.append({'id': r['id'], 'output': text})
        if (i // batch_size) % 10 == 0:
            print(f'  {i+len(batch)}/{len(rows)} done')
    return results

print('Inference function defined')

In [ ]:
# Inline scorer (mirrors score.py logic)
KEYS = ['category','priority','amount_paise','txn_date','txn_id','channel','language','needs_human']

def parse_pred(text):
    try:
        obj = json.loads(text.strip())
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

def score_preds(gold_rows, pred_list):
    preds = {p['id']: p['output'] for p in pred_list}
    n, valid, exact, field_hits = 0, 0, 0, defaultdict(int)
    for g in gold_rows:
        gold = json.loads(g['messages'][-1]['content'])
        pred = parse_pred(preds.get(g['id'], ''))
        n += 1
        if pred is None: continue
        valid += 1
        hits = {k: type(pred.get(k)) is type(gold[k]) and pred.get(k)==gold[k] for k in KEYS}
        all_ok = all(hits.values()) and set(pred)==set(KEYS)
        exact += all_ok
        for k in KEYS: field_hits[k] += hits[k]
    mfa = sum(field_hits[k] for k in KEYS) / (n * len(KEYS))
    print(f'  n={n}  json_valid={valid/n:.1%}  exact_match={exact/n:.1%}  mean_field_acc={mfa:.1%}')
    for k in KEYS: print(f'    {k:<16} {field_hits[k]/n:.1%}')
    return {'n':n,'json_valid':valid/n,'exact_match':exact/n,'mean_field_acc':mfa,
            'fields':{k:field_hits[k]/n for k in KEYS}}

print('Scorer defined')

In [ ]:
print('=== Baseline B1: Zero-shot (no rules in prompt) ===')
b1_preds = run_inference(base_model, base_tok, dev_rows, SYSTEM_SHORT)
b1_score = score_preds(dev_rows, b1_preds)
# Save
with open(f'{DRIVE_DIR}/b1_dev_preds.jsonl','w') as f:
    for r in b1_preds: f.write(json.dumps(r)+'\n')

In [ ]:
print('=== Baseline B2: Rules in system prompt ===')
b2_preds = run_inference(base_model, base_tok, dev_rows, RULES_SYSTEM)
b2_score = score_preds(dev_rows, b2_preds)
with open(f'{DRIVE_DIR}/b2_dev_preds.jsonl','w') as f:
    for r in b2_preds: f.write(json.dumps(r)+'\n')

---
## Part C — Fine-tune Pipeline

In [ ]:
def make_dataset(rows, tokenizer, max_seq):
    """
    Format each row using the model's chat template.
    Returns a HuggingFace Dataset with 'text' column.
    """
    texts = []
    for r in rows:
        text = tokenizer.apply_chat_template(
            r['messages'], tokenize=False, add_generation_prompt=False)
        texts.append({'text': text})
    return Dataset.from_list(texts)

print('Dataset builder defined')

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
import psutil

def train_and_eval(
    run_name,
    train_rows,
    dev_rows,
    lora_r=16,
    lora_alpha=32,
    load_in_4bit=True,
    num_epochs=3,
    lr=2e-4,
    per_device_batch=4,
    grad_accum=4,
    save_dir=None,
):
    print(f'\n{"="*60}')
    print(f'RUN: {run_name}')
    print(f'  train={len(train_rows)}  r={lora_r}  alpha={lora_alpha}  4bit={load_in_4bit}')
    print(f'  epochs={num_epochs}  lr={lr}  batch={per_device_batch}x{grad_accum}')

    t0 = time.time()

    # 1. Load fresh model
    model, tok = FastLanguageModel.from_pretrained(
        MODEL_ID, max_seq_length=MAX_SEQ, load_in_4bit=load_in_4bit, dtype=None)

    # 2. Add LoRA adapter
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=['q_proj','k_proj','v_proj','o_proj',
                        'gate_proj','up_proj','down_proj'],
        lora_dropout=0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
    )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable params: {n_params/1e6:.1f} M')

    # 3. Build dataset
    ds = make_dataset(train_rows, tok, MAX_SEQ)

    # 4. Trainer — loss on assistant tokens only via SFTTrainer's dataset_text_field
    # With Unsloth + SFTTrainer, the chat-formatted text with EOS ensures
    # the trainer computes loss only on assistant completions.
    cfg = SFTConfig(
        output_dir=f'/tmp/{run_name}',
        num_train_epochs=num_epochs,
        per_device_train_batch_size=per_device_batch,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        fp16=not load_in_4bit,
        bf16=False,
        optim='paged_adamw_8bit',
        logging_steps=20,
        save_strategy='no',
        seed=SEED,
        max_seq_length=MAX_SEQ,
        dataset_text_field='text',
        packing=False,
    )
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=cfg)
    trainer.train()

    # 5. Peak GPU memory
    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    torch.cuda.reset_peak_memory_stats()

    # 6. Evaluate on dev
    FastLanguageModel.for_inference(model)
    preds = run_inference(model, tok, dev_rows, SYSTEM_SHORT)
    result = score_preds(dev_rows, preds)

    t1 = time.time()
    mins = (t1 - t0) / 60
    result['peak_gpu_mb'] = peak_mb
    result['train_minutes'] = mins
    result['run_name'] = run_name

    print(f'  Peak GPU: {peak_mb:.0f} MB   Time: {mins:.1f} min')

    # 7. Save adapter
    adapter_path = save_dir or f'{DRIVE_DIR}/{run_name}_adapter'
    model.save_pretrained(adapter_path)
    tok.save_pretrained(adapter_path)
    print(f'  Adapter saved → {adapter_path}')

    # 8. Save dev preds
    pred_path = f'{DRIVE_DIR}/{run_name}_dev_preds.jsonl'
    with open(pred_path, 'w') as f:
        for r in preds: f.write(json.dumps(r)+'\n')

    return model, tok, result, preds

print('train_and_eval() defined')

---
## Part D — Controlled Experiments

In [ ]:
# E1: Raw training data, baseline config (4-bit QLoRA, r=16)
_, _, e1_result, _ = train_and_eval(
    'E1_raw_data', train_raw, dev_rows,
    lora_r=16, lora_alpha=32, load_in_4bit=True, num_epochs=3, lr=2e-4)

In [ ]:
# E2: Cleaned training data, same config — this is the reference fine-tune
best_model, best_tok, e2_result, e2_preds = train_and_eval(
    'E2_clean_data', train_clean, dev_rows,
    lora_r=16, lora_alpha=32, load_in_4bit=True, num_epochs=3, lr=2e-4,
    save_dir=f'{DRIVE_DIR}/adapter')

In [ ]:
# E3: 16-bit LoRA (bf16) vs E2 QLoRA — same data, r, epochs
_, _, e3_result, _ = train_and_eval(
    'E3_16bit_lora', train_clean, dev_rows,
    lora_r=16, lora_alpha=32, load_in_4bit=False, num_epochs=3, lr=2e-4)

In [ ]:
# E4: Higher rank r=32 — same data, 4-bit, epochs
_, _, e4_result, _ = train_and_eval(
    'E4_rank32', train_clean, dev_rows,
    lora_r=32, lora_alpha=64, load_in_4bit=True, num_epochs=3, lr=2e-4)

In [ ]:
# E5 (your idea): 5 epochs on cleaned data — longer training
_, _, e5_result, _ = train_and_eval(
    'E5_5epochs', train_clean, dev_rows,
    lora_r=16, lora_alpha=32, load_in_4bit=True, num_epochs=5, lr=2e-4)

In [ ]:
# Print experiment table
results = {
    'B1 zero-shot':        b1_score,
    'B2 rules-in-prompt':  b2_score,
    'E1 raw data':         e1_result,
    'E2 clean data (ref)': e2_result,
    'E3 16-bit LoRA':      e3_result,
    'E4 rank=32':          e4_result,
    'E5 5 epochs':         e5_result,
}

print(f'\n{"Run":<22} {"Exact":>7} {"MFA":>7} {"GPU MB":>8} {"Mins":>6}')
print('-' * 55)
for name, r in results.items():
    em   = r.get('exact_match', 0)
    mfa  = r.get('mean_field_acc', 0)
    gpu  = r.get('peak_gpu_mb', 0)
    mins = r.get('train_minutes', 0)
    print(f'{name:<22} {em:>7.1%} {mfa:>7.1%} {gpu:>8.0f} {mins:>6.1f}')

# Save table JSON
with open(f'{DRIVE_DIR}/experiment_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nResults saved to Drive')

---
## Part E — Error Analysis

In [ ]:
# Use E2 (clean data, baseline config) predictions as 'best model'
# Identify all failures: exact_match = False
failures = []
preds_map = {p['id']: p['output'] for p in e2_preds}

for g in dev_rows:
    gold  = json.loads(g['messages'][-1]['content'])
    raw   = preds_map.get(g['id'], '')
    pred  = parse_pred(raw)
    if pred is None:
        failures.append({'id': g['id'], 'ticket': g['messages'][1]['content'],
                         'gold': gold, 'pred': None, 'wrong_fields': ['INVALID JSON'], 'raw': raw})
        continue
    wrong = [k for k in KEYS if not (type(pred.get(k)) is type(gold[k]) and pred.get(k)==gold[k])]
    if wrong or set(pred) != set(KEYS):
        failures.append({'id': g['id'], 'ticket': g['messages'][1]['content'],
                         'gold': gold, 'pred': pred, 'wrong_fields': wrong, 'raw': raw})

print(f'Total failures: {len(failures)} / {len(dev_rows)}')

# Field failure counts
from collections import Counter
field_fails = Counter()
for f in failures:
    for k in f['wrong_fields']: field_fails[k] += 1
print('\nField failure counts:')
for k, v in field_fails.most_common():
    print(f'  {k:<16} {v}')

In [ ]:
# Print first 15 failures for analysis
for i, f in enumerate(failures[:15]):
    print(f'\n--- Failure {i+1}: {f["id"]}  wrong={f["wrong_fields"]} ---')
    # Truncate ticket
    ticket_lines = f['ticket'].split('\n')
    print('  Ticket:', ' '.join(ticket_lines[:6])[:200])
    print('  Gold: ', f['gold'])
    print('  Pred: ', f['pred'])

---
## Generate Test Predictions (Submission)

In [ ]:
# Use best model (E2 or whichever scored highest)
# If session restarted, reload the adapter:
# from unsloth import FastLanguageModel
# best_model, best_tok = FastLanguageModel.from_pretrained(
#     f'{DRIVE_DIR}/adapter', max_seq_length=MAX_SEQ, load_in_4bit=True)
# FastLanguageModel.for_inference(best_model)

print('Generating test predictions...')
# test_inputs has only system + user messages (no assistant)
test_preds = run_inference(best_model, best_tok, test_rows, SYSTEM_SHORT)

out_path = 'predictions_test.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for r in test_preds:
        f.write(json.dumps({'id': r['id'], 'output': r['output']}) + '\n')

# Also save to Drive
import shutil
shutil.copy(out_path, f'{DRIVE_DIR}/{out_path}')
print(f'Wrote {len(test_preds)} predictions → {out_path}')

# Quick sanity check: how many are valid JSON?
valid = sum(1 for r in test_preds if parse_pred(r['output']) is not None)
print(f'Valid JSON: {valid}/{len(test_preds)} ({valid/len(test_preds):.1%})')

In [ ]:
# Download predictions_test.jsonl to your machine
from google.colab import files
files.download('predictions_test.jsonl')

---
## Reload adapter in a fresh session

```python
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(
    '/content/drive/MyDrive/nimbuspay_triage/adapter',
    max_seq_length=512, load_in_4bit=True)
FastLanguageModel.for_inference(model)
```